# 06 - Build the web app folder (PN leads, AW backs up)

**Plain words:** the website is just a small Gradio app that calls the same `predict()` function as the command line.
This notebook builds the folder you upload to a free **Hugging Face Space** (the public link).

**Do it twice**
1. **Now, with a PLACEHOLDER model** (untrained, the page says TEST VERSION). This proves the deployment works early, so there are no surprises at the end.
2. **Later, with the real final model** (GW finishes notebook 03 CELL 10 first).

**One-time:** create a free account at huggingface.co, then *New > Space*: name it, SDK **Gradio**, hardware **CPU basic (free)**, visibility **Public**.

In [ ]:
# CELL 1 - run this first. It downloads the project code into this Colab session.
import os, subprocess, sys
REPO_URL = "https://github.com/andrinawatsemba/DEEPFAKE_CV_INVESTIGATION.git"
CODE = "/content/repo"
if not os.path.isdir(CODE + "/src"):
    for branch in ("main", "gw/lean-pipeline"):          # main first; the branch is only a fallback
        subprocess.run(["rm", "-rf", CODE])
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", branch, REPO_URL, CODE])
        if r.returncode == 0 and os.path.exists(CODE + "/src/colab_setup.py"):
            print("Got the code from branch:", branch)
            break
else:
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"])
sys.path.insert(0, CODE + "/src")
print("Code ready.")


In [ ]:
# CELL 2 - connect Drive and pick the mode
!pip -q install gradio
import os, shutil, subprocess
from colab_setup import setup
ROOT, REPO = setup()
USE_PLACEHOLDER = True      # True = untrained test model. Change to False once models/final.pt exists on Drive.

In [ ]:
# CELL 3 - build the folder
os.makedirs(f"{REPO}/models", exist_ok=True)
if not USE_PLACEHOLDER:
    for f in ("final.pt", "final.json"):
        shutil.copy(f"{ROOT}/models/{f}", f"{REPO}/models/{f}")
args = ["python", f"{REPO}/tools/make_space_folder.py"] + (["--placeholder"] if USE_PLACEHOLDER else [])
print(subprocess.run(args, capture_output=True, text=True).stdout)
shutil.make_archive("/content/space_build", "zip", f"{REPO}/space_build")
print(os.listdir(f"{REPO}/space_build"))

In [ ]:
# CELL 4 - test the app function here BEFORE uploading (use any photo from Drive)
import sys, glob
sys.path.insert(0, f"{REPO}/space_build")
import importlib, app as webapp
importlib.reload(webapp)
sample = sorted(glob.glob(f"{ROOT}/public_data/openfake/*"))[:1]
print(webapp.analyse(sample[0])[0] if sample else "put a photo path here")

In [ ]:
# CELL 5 - download the folder as a zip
from google.colab import files
files.download("/content/space_build.zip")

## Upload to Hugging Face (step by step)
1. Unzip the download on your computer.
2. Open your Space > **Files** tab > **Add file > Upload files**. Drag **everything inside** the unzipped folder (`app.py`, `README.md`, `requirements.txt`, `src/`, `models/`). The `.pt` file is big, so Hugging Face handles it automatically.
3. Click **Commit changes**. The Space shows *Building*, then *Running* (2 to 5 minutes).
4. Open the public link on your phone, on mobile data, in a private window. Upload 5 different photos.
5. Paste the link in the group chat and into the README.

**Before the examination:** repeat step 4 the day before. Free Spaces go to sleep when unused, so the first request can be slow: open it early.